# Gemma 4 (8B) QLoRA DPO Training on Colab Free Tier
This notebook is specifically configured to fit an 8B model into the 16GB VRAM limit of a Free Tier Google Colab Tesla T4 GPU.

### Step 1: Install Dependencies
We need to install specific unsloth, trl, and PEFT libraries to handle 4-bit quantization.

In [ ]:
!pip install -q -U bitsandbytes transformers peft accelerate trl datasets

### Step 2: Load Model in 4-bit Precision
This compresses the 16GB base model down to ~4.5GB.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import DPOTrainer
from datasets import load_dataset

# 1. Choose Model Repository
model_id = "google/gemma-4-8b-it"

# 2. The 4-Bit Quantization Config (Saves ~10GB of VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

print(f"Loading {model_id} in 4-bit...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Ensure EOS token is used as pad token to prevent formatting errors
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    use_cache=False # Crucial for saving memory during training
)

model = prepare_model_for_kbit_training(model)

### Step 3: Configure LoRA and Training Parameters
We use a highly restrictive batch size and gradient accumulation to simulate learning without spiking RAM.

In [ ]:
# 3. LoRA Config: Only train ~2% of the parameters
lora_config = LoraConfig(
    r=8, 
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# 4. Training Arguments explicitly designed to heavily restrict GPU Usage
training_args = TrainingArguments(
    output_dir="./triage-dpo-gemma",
    per_device_train_batch_size=1,       # MUST BE 1 to avoid OOM
    gradient_accumulation_steps=4,       # Simulates a batch size of 4
    optim="paged_adamw_32bit",           # Offloads optimizer states to CPU RAM if needed
    save_steps=50,
    logging_steps=10,
    learning_rate=2e-5,
    bf16=True,                           # Faster and lighter than FP32
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
)

### Step 4: Load Dataset and Start Training

In [ ]:
# Load your DPO dataset here (replace with your huggingface dataset repo)
# dataset = load_dataset("your_username/your_triage_dataset")

# Initialize DPO Trainer
# trainer = DPOTrainer(
#     model=model,
#     ref_model=None, # Peft handles reference automatically
#     args=training_args,
#     beta=0.1,
#     train_dataset=dataset['train'], 
#     tokenizer=tokenizer,
#     max_length=1024,        # Force short sequences to prevent memory spiking
#     max_prompt_length=512,  # Keep prompts concise
# )

# trainer.train()